# ray-parametric-form — worked example 2: Evaluate a batch of rays at the same parameter value

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ray-parametric-form`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

When you have a batch of `B` rays stored as shape `(B, 2, 3)`, you can evaluate all of them at the same scalar parameter `u` in a single vectorized operation: `O + u * D` where `O = rays[:, 0]` and `D = rays[:, 1]` are both shape `(B, 3)`. The scalar `u` broadcasts across all `B` rays simultaneously.

## Worked solution

**Step 1 — Slice origins and directions.** `O = rays[:, 0]` extracts row 0 from each `(2, 3)` block, giving shape `(B, 3)`. Similarly `D = rays[:, 1]` gives direction vectors.

**Step 2 — Apply the formula.** `O + u * D` with scalar `u` broadcasts: `u * D` scales each direction vector, then adds the corresponding origin. The result has shape `(B, 3)`.

**Step 3 — Verify shapes.** We check that the output shape is `(B, 3)` and that the first result (for ray 0) matches the single-ray evaluation.

**Step 4 — Geometric check.** We set `u=0` and confirm the output equals the origins `rays[:, 0]` exactly.

In [ ]:
import torch as t

def eval_ray_batch(rays: t.Tensor, u: float) -> t.Tensor:
    """Evaluate R(u) = O + u*D for a batch of rays at the same u."""
    O = rays[:, 0]   # (B, 3)
    D = rays[:, 1]   # (B, 3)
    return O + u * D  # (B, 3)

t.manual_seed(62)
B = 5
# Each ray: random origin and direction
origins    = t.randn(B, 3)
directions = t.randn(B, 3)
rays = t.stack([origins, directions], dim=1)  # (B, 2, 3)

# Evaluate at u=1.5
u = 1.5
points = eval_ray_batch(rays, u)
print(f'Output shape: {points.shape}')       # (5, 3)

# Verify against single-ray formula for ray index 2
O2, D2 = rays[2, 0], rays[2, 1]
expected_2 = O2 + u * D2
print(f'Ray 2 matches: {t.allclose(points[2], expected_2)}')  # True

# At u=0, all points should equal origins
points_at_0 = eval_ray_batch(rays, 0.0)
print(f'u=0 equals origins: {t.equal(points_at_0, origins)}')  # True